In [1]:
from datetime import datetime
from typing import Literal, Optional
import pybamm
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from netlist_utils import setup_circuit
from battery import LiionBatteryPack, PybammBattery, CLCBattery
import vessim as vs

import nest_asyncio
nest_asyncio.apply()

In [2]:
class CarbonAwareController(vs.Controller):
    def __init__(self, power_meter: vs.MockSignal, ci: vs.HistoricalSignal):
        super().__init__()
        self.power_meter = power_meter
        self.ci = ci

    def step(self, time: datetime, p_delta: float, e_delta: float, state: dict) -> None:
        self.set_parameters["policy:min_soc"] = 0.5 if (time.hour < 11 or time.hour > 13) and state["storage"]["soc"] >= 0.5 else 0.3
        self.set_parameters["policy:charge_power"] = 40 if (time.hour <= 4 or time.hour >= 23) and self.ci.now(time) <= 60 else 0
        #self.power_meter.set_value(-9/5 * 16 if 3 <= time.hour < 9 else -18/5 * 16)
        self.power_meter.set_value(-18/5 * 16 if time.hour < 9 else 18/5 * 16)

In [3]:
class MinMaxSocPolicy(vs.MicrogridPolicy):
    def __init__(self, mode: Literal["grid-connected", "islanded"] = "grid-connected", charge_power: Optional[float] = None, min_soc = 0, max_soc=1):
        self.mode = mode
        self.charge_power = charge_power if charge_power else 0.0
        self.min_soc = min_soc
        self.max_soc = max_soc

    def apply(self, p_delta: float, duration: int, storage: Optional[vs.Storage] = None) -> float:
        energy_delta = p_delta * duration
        if storage and ((self.min_soc >= storage.soc() and p_delta <= 0) or (self.max_soc <= storage.soc() and p_delta >= 0)):
            storage.update(0.0, duration)
            if self.mode == "islanded":
                if energy_delta < 0.0:
                    raise RuntimeError("Not enough energy available to operate in islanded mode.")
                energy_delta = 0.0
        elif self.mode == "grid-connected" and storage is not None:
            if self.charge_power:
                energy_delta -= storage.update(self.charge_power, duration)
            else:
                energy_delta -= storage.update(p_delta, duration)
        elif self.mode == "islanded":
            if storage:
                energy_delta -= storage.update(p_delta, duration)
            if energy_delta < 0.0:
                raise RuntimeError("Not enough energy available to operate in islanded mode.")
            energy_delta = 0.0
        return energy_delta

    def state(self) -> dict:
        return {
            "mode": self.mode,
            "charge_power": self.charge_power,
            "min_soc": self.min_soc
        }


In [4]:
def default_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.8 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model_type=pybamm.lithium_ion.SPMe,
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("Chen2020"),
        output_variables=["Current [A]", "Voltage [V]", "Power [W]", "Discharge energy [W.h]"],
    )
    liion_battery_pack = LiionBatteryPack(
        model=pybamm.lithium_ion.SPMe(),
        netlist = setup_circuit(Np=np, Ns=ns, Rb=1.5e-3, Rc=1e-2, Ri=5e-2, V=4.0, I=5.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("Chen2020"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [5]:

def degredation_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.8 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model=pybamm.lithium_ion.SPMe(
            options={
                "SEI": "solvent-diffusion limited",
                "SEI porosity change": "true",
                "lithium plating": "partially reversible",
                "lithium plating porosity change": "true",  # alias for "SEI porosity change"
                "particle mechanics": ("swelling and cracking", "swelling only"),
                "SEI on cracks": "true",
                "loss of active material": "stress-driven",
                "operating mode": "power",
            }
        ),
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("OKane2022"),
        output_variables=["Voltage [V]", "Current [A]", "Power [W]"]
    )
    liion_battery_pack = LiionBatteryPack(
        model=pybamm.lithium_ion.SPMe(
            options={
                "SEI": "solvent-diffusion limited",
                "SEI porosity change": "true",
                "lithium plating": "partially reversible",
                "lithium plating porosity change": "true",  # alias for "SEI porosity change"
                "particle mechanics": ("swelling and cracking", "swelling only"),
                "SEI on cracks": "true",
                "loss of active material": "stress-driven",
            }
        ),
        netlist = setup_circuit(Np=np, Ns=ns, Rb=1.5e-3, Rc=1e-2, Ri=5e-2, V=4.0, I=5.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("OKane2022"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [ ]:
STEP_SIZE = 10
POWER = -18.8 / 5
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
for battery in default_battery_models(initial_soc=1, np=1, ns=1, step_size=STEP_SIZE)[:2]:
    monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}.csv")
    environment.add_microgrid(
        actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=POWER))],
        storage=battery,
        controllers=[monitor],
        step_size=STEP_SIZE,
    )

environment.run(until=5.1 * 3600)

In [ ]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=300,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend={"xanchor": "right", "x": 1}
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Time}$", ticksuffix="h", range=[0, 5.07]
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{State-of-Charge}$", ticksuffix="%", range=[0, 103]
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, name=battery, mode="lines")
    )
fig.write_image("../thesis/Images/discharge_energy.pdf", "pdf")
fig.show()

In [6]:
STEP_SIZE = 10
C_RATES_DISCHARGE = [0.1, 0.2, 0.3, 0.5, 1.0, 2.0]
for c_rate in C_RATES_DISCHARGE:
    power = -18.78 * c_rate
    environment = vs.Environment(sim_start="2022-07-03 09:00:00")
    for battery in default_battery_models(initial_soc=1, np=1, ns=1, step_size=STEP_SIZE)[:3]:
        monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}_Discharge{c_rate}C.csv")
        environment.add_microgrid(
            actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=power))],
            storage=battery,
            controllers=[monitor],
            step_size=STEP_SIZE,
        )
    if c_rate < 1.5:
        environment.run(until=(1 / c_rate) * 3800)
    else:
        environment.run(until=2 / 3 * 3800)

2024-07-04 15:36:30.125 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:SimpleBattery" ...
2024-07-04 15:36:30.126 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-0" ...
2024-07-04 15:36:30.127 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-0" ...
2024-07-04 15:36:30.128 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-0" ...
2024-07-04 15:36:30.129 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:CLCBattery" ...
2024-07-04 15:36:30.130 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-1" ...
2024-07-04 15:36:30.131 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-1" ...
2024-07-04 15:36:30.131 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-1" ...
2024-07-04 15:36:30.132 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:PybammBattery" ...
2024-07-04 15:36:30.133 | INFO     | mosaik.sc

Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.


 14%|█▍        | 363/2533.333333333333 [00:00<00:02, 884.74steps/s]

Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.


 22%|██▏       | 560/2533.333333333333 [00:00<00:02, 926.12steps/s]

...

Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.


 89%|████████▉ | 2262/2533.333333333333 [00:02<00:00, 764.01steps/s]

Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.


100%|█████████▉| 2533.0/2533.333333333333 [00:03<00:00, 840.95steps/s]
2024-07-04 15:39:02.789 | INFO     | mosaik.scenario:run:708 - Simulation finished successfully.


Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.
Discharging current exceeds maximum discharge rate.


In [7]:
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=tuple(f"C-Rate {i}" for i in C_RATES_DISCHARGE))
fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
showlegend = True
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery"]
colors = ["Blue", "Red", "Green", "Purple"]
for i, c_rate in enumerate(C_RATES_DISCHARGE):
    col = i % 2 + 1
    row = i // 2 + 1
    for battery, color in zip(batteries, colors[:3]):
        df = pd.read_csv(f"results/{battery}_Discharge{c_rate}C.csv")
        fig.add_trace(
            go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, showlegend=showlegend, line_color=color, name=battery), row=row, col=col
        )
    fig.update_xaxes(row=row, col=col, title="", ticksuffix="h")
    fig.update_yaxes(row=row, col=col, title="$\\text{State-of-Charge}$", ticksuffix = "%")
    showlegend = False
fig.show()

In [8]:
STEP_SIZE = 10
C_RATES_CHARGE = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
for c_rate in C_RATES_CHARGE:
    power = 18.78 * c_rate
    environment = vs.Environment(sim_start="2022-07-03 09:00:00")
    for battery in default_battery_models(initial_soc=0, np=1, ns=1, step_size=STEP_SIZE)[:3]:
        monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}_Charge{c_rate}C.csv")
        environment.add_microgrid(
            actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=power))],
            storage=battery,
            controllers=[monitor],
            step_size=STEP_SIZE,
        )
    if c_rate < 0.7:
        environment.run(until=(1 / c_rate) * 4200)
    else:
        environment.run(until=1 / 0.7 * 4200)

2024-07-04 15:39:06.340 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:SimpleBattery" ...
2024-07-04 15:39:06.341 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-0" ...
2024-07-04 15:39:06.343 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-18" ...
2024-07-04 15:39:06.344 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-0" ...
2024-07-04 15:39:06.345 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:CLCBattery" ...
2024-07-04 15:39:06.345 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-1" ...
2024-07-04 15:39:06.346 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-19" ...
2024-07-04 15:39:06.347 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-1" ...
2024-07-04 15:39:06.348 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:PybammBattery" ...
2024-07-04 15:39:06.348 | INFO     | mosaik.

Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum chargin

  8%|▊         | 457/6000.0 [00:00<00:05, 1096.13steps/s]

Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum chargin

 11%|█▏        | 676/6000.0 [00:00<00:05, 1058.17steps/s]

...

Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.


 99%|█████████▉| 5933/6000.0 [00:07<00:00, 614.59steps/s]

Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.
Charging current exceeds maximum charging rate.


100%|██████████| 6000.0/6000.0 [00:07<00:00, 779.50steps/s]
2024-07-04 15:42:15.972 | INFO     | mosaik.scenario:run:708 - Simulation finished successfully.


Charging current exceeds maximum charging rate.


In [9]:
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=tuple(f"C-Rate {i}" for i in C_RATES_CHARGE))
fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
showlegend = True
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery"]
colors = ["Blue", "Red", "Green", "Purple"]
for i, c_rate in enumerate(C_RATES_CHARGE):
    col = i % 2 + 1
    row = i // 2 + 1
    for battery, color in zip(batteries, colors[:3]):
        df = pd.read_csv(f"results/{battery}_Charge{c_rate}C.csv")
        fig.add_trace(
            go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, showlegend=showlegend, line_color=color, name=battery), row=row, col=col
        )
    fig.update_xaxes(row=row, col=col, title="", ticksuffix="h")
    fig.update_yaxes(row=row, col=col, title="$\\text{State-of-Charge}$", ticksuffix = "%")
    showlegend = False
fig.show()

In [ ]:
STEP_SIZE = 30
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
for battery in default_battery_models(initial_soc=0.7, np=4, ns=4, step_size=STEP_SIZE):
    policy = MinMaxSocPolicy(min_soc=0.5)
    power_meter = vs.MockSignal(value=-18 / 5 * 16)
    ci = vs.HistoricalSignal(actual=pd.read_csv("data/watttime2022_caiso-north_actual.csv", index_col=0))
    monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}.csv", grid_signals={"carbon-intensity": ci})
    ca_controller = CarbonAwareController(power_meter, ci=ci)
    environment.add_microgrid(
        actors=[
            vs.Actor(name=f"MockServer:{type(battery).__name__}", signal=power_meter),
            #vs.Actor(name=f"MockSolar:{type(battery).__name__}", signal=vs.HistoricalSignal.load("solcast2022_global", column="San Francisco", params={"scale": 180})),
        ],
        policy=policy,
        storage=battery,
        controllers=[monitor, ca_controller],
        step_size=STEP_SIZE,  # Global step size (can be overridden by actors or controllers)
    )

environment.run(until=24 * 3600)

In [ ]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=df.index, y=df["storage.soc"], name=battery, mode="lines")
    )
fig.show()

In [ ]:
STEP_SIZE = 10
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
for battery in init_battery_models(initial_soc=0, np=4, ns=4, step_size=STEP_SIZE):
    monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}.csv")
    environment.add_microgrid(
        actors=[MockActor(p=120)],
        storage=battery,
        controllers=[monitor],
        step_size=STEP_SIZE,
    )

environment.run(until=4 * 3600)

In [ ]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=[i * STEP_SIZE for i in range(len(df.index))], y=df["storage.soc"], name=battery, mode="lines")
    )
fig.show()